In [ ]:
# ============================================================
# AI LEARNING STUDIO - GOOGLE COLAB
# Part 1: Setup, dependencies, API keys, and AI functions
# ============================================================

import os
import re
import sys
import time
import io
import subprocess
import urllib.request

from google.colab import userdata

# Install required packages
packages = [
    "streamlit",
    "openai",
    "huggingface_hub",
    "requests",
    "diffusers",
    "transformers",
    "accelerate",
    "safetensors",
]

for package in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", package],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if result.returncode != 0:
        print("Warning: installation issue:", package)
        print(result.stdout[-1500:])

# Read Colab Secrets
def get_secret(name):
    try:
        return userdata.get(name) or ""
    except Exception:
        return ""

ENV = os.environ.copy()
ENV["GROQ_API_KEY"] = get_secret("GROQ_API_KEY")
ENV["HF_API_KEY"] = get_secret("HF_API_KEY")

if not ENV["GROQ_API_KEY"] and not ENV["HF_API_KEY"]:
    print("WARNING: Add GROQ_API_KEY or HF_API_KEY in Colab Secrets.")

# Save keys to the notebook process environment for this cell
os.environ.update({
    "GROQ_API_KEY": ENV["GROQ_API_KEY"],
    "HF_API_KEY": ENV["HF_API_KEY"],
})

APP_CODE = r'''
import os
import re
import io
import streamlit as st
from openai import OpenAI
from huggingface_hub import InferenceClient

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
HF_API_KEY = os.environ.get("HF_API_KEY", "")

GROQ_URL = "https://api.groq.com/openai/v1"

GROQ_MODELS = [
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant",
    "openai/gpt-oss-20b",
    "openai/gpt-oss-120b",
    "qwen/qwen3-32b",
]

HF_MODELS = ["meta-llama/Llama-3.1-8B-Instruct"]


def generate_response(prompt, temperature=0.3, max_tokens=1024):
    provider = st.session_state.get("provider", "Groq")

    if provider == "Groq":
        if not GROQ_API_KEY:
            return "Error: Add GROQ_API_KEY in Colab Secrets."

        client = OpenAI(
            api_key=GROQ_API_KEY,
            base_url=GROQ_URL,
            timeout=45,
            max_retries=1,
        )

        errors = []

        for model in GROQ_MODELS:
            try:
                result = client.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=temperature,
                    max_tokens=max_tokens,
                )
                return result.choices[0].message.content or "Empty response."
            except Exception as e:
                errors.append(f"{model}: {e}")

        return "Groq request failed:\n" + "\n".join(errors)

    if not HF_API_KEY:
        return "Error: Add HF_API_KEY in Colab Secrets."

    try:
        client = InferenceClient(
            model=HF_MODELS[0],
            token=HF_API_KEY,
            timeout=45,
        )
        result = client.chat_completion(
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return result.choices[0].message.content or "Empty response."
    except Exception as e:
        return f"Hugging Face request failed: {e}"


MATH_SYSTEM = r"""
You are Math Mastermind. Explain solutions step by step.
Verify the result when possible. Use LaTeX for all mathematics.
Use $...$ for inline math and $$...$$ for display equations.
Use \times for multiplication. Clearly state the final answer.
"""

TEACH_SYSTEM = r"""
You are a helpful AI Teaching Assistant.
Explain concepts clearly and accurately with examples when useful.
Use $...$ for inline mathematics and $$...$$ for display equations.
"""


def fix_math(text):
    text = re.sub(
        r"\\\[(.+?)\\\]",
        lambda m: "$$" + m.group(1).strip() + "$$",
        text,
        flags=re.S,
    )
    text = re.sub(
        r"\\\((.+?)\\\)",
        lambda m: "$" + m.group(1).strip() + "$",
        text,
        flags=re.S,
    )
    return text


def render_answer(text):
    text = fix_math(text)
    display_re = re.compile(r"\$\$(.+?)\$\$", re.S)
    position = 0

    for match in display_re.finditer(text):
        prose = text[position:match.start()]
        if prose.strip():
            st.markdown(prose)
        st.latex(match.group(1).strip())
        position = match.end()

    if text[position:].strip():
        st.markdown(text[position:])


def export_history(history):
    content = ""
    for i, item in enumerate(history, 1):
        content += f"Q{i}: {item['question']}\n"
        content += f"A{i}: {item['answer']}\n\n"
    return content.encode("utf-8")


def teaching_answer(question):
    return generate_response(
        TEACH_SYSTEM + "\n\nQuestion: " + question,
        temperature=0.3,
    )


def math_answer(question, level):
    return generate_response(
        MATH_SYSTEM
        + f"\nDifficulty: {level}\nProblem: {question}",
        temperature=0.1,
    )
'''

APP_CODE += r'''

# ============================================================
# FEATURE 1: AI TEACHING ASSISTANT
# ============================================================

def run_ai_teaching_assistant():
    st.title("🤖 AI Teaching Assistant")
    st.caption("Ask questions and receive clear explanations.")

    st.session_state.setdefault("history_ata", [])

    col1, col2 = st.columns([1, 2])

    if col1.button("🧹 Clear", key="clear_ata"):
        st.session_state.history_ata = []
        st.rerun()

    if st.session_state.history_ata:
        col2.download_button(
            "📄 Export",
            data=export_history(st.session_state.history_ata),
            file_name="teaching_assistant.txt",
            mime="text/plain",
        )

    with st.form("ata_form"):
        question = st.text_area(
            "Your question",
            placeholder="Explain photosynthesis...",
        )
        submitted = st.form_submit_button("Ask AI")

    if submitted:
        if not question.strip():
            st.warning("Enter a question.")
        else:
            with st.spinner("Thinking..."):
                answer = teaching_answer(question.strip())

            st.session_state.history_ata.append({
                "question": question.strip(),
                "answer": answer,
            })
            st.rerun()

    for i, item in enumerate(st.session_state.history_ata, 1):
        with st.container(border=True):
            st.markdown(f"**Question {i}:**")
            st.text(item["question"])
            st.markdown("**Answer:**")
            render_answer(item["answer"])


# ============================================================
# FEATURE 2: MATH MASTERMIND
# ============================================================

def run_math_mastermind():
    st.title("🧮 Math Mastermind")
    st.caption("Get step-by-step math solutions.")

    st.session_state.setdefault("history_mm", [])

    col1, col2 = st.columns([1, 2])

    if col1.button("🧹 Clear", key="clear_mm"):
        st.session_state.history_mm = []
        st.rerun()

    if st.session_state.history_mm:
        col2.download_button(
            "📄 Export",
            data=export_history(st.session_state.history_mm),
            file_name="math_solutions.txt",
            mime="text/plain",
        )

    with st.form("math_form"):
        question = st.text_area(
            "Math problem",
            placeholder="Solve 2x + 5 = 17",
        )
        level = st.selectbox(
            "Difficulty",
            ["Basic", "Intermediate", "Advanced"],
            index=1,
        )
        submitted = st.form_submit_button("🧮 Solve")

    if submitted:
        if not question.strip():
            st.warning("Enter a math problem.")
        else:
            with st.spinner("Solving..."):
                answer = math_answer(question.strip(), level)

            st.session_state.history_mm.insert(0, {
                "question": question.strip(),
                "answer": answer,
                "difficulty": level,
            })
            st.rerun()

    for i, item in enumerate(st.session_state.history_mm, 1):
        with st.container(border=True):
            st.markdown(f"**Problem {i}:**")
            st.text(item["question"])
            st.caption("Difficulty: " + item["difficulty"])
            render_answer(item["answer"])


# ============================================================
# FEATURE 3: SAFE AI IMAGE GENERATOR
# ============================================================

IMAGE_MODEL = "stabilityai/sdxl-turbo"

STYLES = {
    "None": "",
    "Photorealistic": ", photorealistic, natural lighting, high detail",
    "Digital art": ", digital art, vibrant colors, detailed",
    "Cartoon": ", friendly cartoon, bold outlines",
    "Watercolor": ", watercolor painting, soft colors",
    "3D render": ", 3D render, soft studio lighting",
}

BLOCKED_RE = re.compile(
    r"\b(nude|naked|nsfw|porn\w*|sexual\w*|erotic\w*|"
    r"fetish|gore|gory|mutilat\w*|dismember\w*|"
    r"behead\w*|tortur\w*|suicide|self[- ]harm|"
    r"terroris\w*|child abuse|loli\w*)\b",
    re.IGNORECASE,
)


def check_prompt_safety(prompt):
    if BLOCKED_RE.search(prompt):
        return False, "Please use a family-friendly image description."

    # Require an available AI provider for moderation.
    verdict = generate_response(
        "Classify this image prompt. Reply with exactly SAFE or UNSAFE. "
        "UNSAFE includes sexual content, graphic violence, self-harm, "
        "hate, illegal instructions, or sexualization of minors.\n"
        "Prompt: " + prompt,
        temperature=0,
        max_tokens=5,
    ).strip().upper()

    if verdict.startswith("SAFE"):
        return True, ""

    if verdict.startswith("UNSAFE"):
        return False, "The prompt was flagged as unsafe."

    return False, "Safety check failed. Check your AI provider and retry."


def make_image(prompt):
    errors = []

    # Local image generation requires a working GPU.
    try:
        import torch

        if torch.cuda.is_available():
            from diffusers import AutoPipelineForText2Image

            pipe = load_image_pipe()

            image = pipe(
                prompt=prompt,
                num_inference_steps=4,
                guidance_scale=0.0,
                width=512,
                height=512,
            ).images[0]

            buffer = io.BytesIO()
            image.save(buffer, format="PNG")
            return buffer.getvalue(), IMAGE_MODEL

        errors.append("No GPU detected in this Colab runtime.")

    except Exception as e:
        errors.append(f"Local generation: {type(e).__name__}: {e}")

    # Optional third-party online fallback.
    try:
        import requests
        import urllib.parse

        url = (
            "https://image.pollinations.ai/prompt/"
            + urllib.parse.quote(prompt)
        )

        response = requests.get(
            url,
            params={
                "width": 768,
                "height": 768,
                "nologo": "true",
                "model": "flux",
            },
            timeout=120,
        )

        if (
            response.ok
            and response.headers.get("content-type", "").startswith("image")
        ):
            return response.content, "pollinations/flux"

        errors.append(f"Online service HTTP {response.status_code}")

    except Exception as e:
        errors.append(f"Online generation: {type(e).__name__}: {e}")

    return None, "Image generation failed:\n" + "\n".join(errors)


@st.cache_resource(show_spinner="Loading local image model...")
def load_image_pipe():
    import torch
    from diffusers import AutoPipelineForText2Image

    pipe = AutoPipelineForText2Image.from_pretrained(
        IMAGE_MODEL,
        torch_dtype=torch.float16,
        variant="fp16",
    )

    return pipe.to("cuda")


def run_safe_ai_image_generator():
    st.title("🎨 Safe AI Image Generator")
    st.caption("Create and download AI-generated images.")

    st.session_state.setdefault("history_img", [])

    if st.button("🧹 Clear image history", key="clear_img"):
        st.session_state.history_img = []
        st.rerun()

    prompt = st.text_area(
        "Describe your image",
        placeholder="A friendly robot teaching math in a sunny classroom",
        key="image_prompt",
    )

    style = st.selectbox("Image style", list(STYLES.keys()))

    if st.button("🎨 Generate image", key="generate_img"):
        if not prompt.strip():
            st.warning("Enter an image description.")
        else:
            with st.spinner("Checking prompt safety..."):
                allowed, message = check_prompt_safety(prompt.strip())

            if not allowed:
                st.error(message)
            else:
                with st.spinner("Generating image..."):
                    data, info = make_image(
                        prompt.strip() + STYLES[style]
                    )

                if data is None:
                    st.error(info)
                else:
                    st.session_state.history_img.insert(0, {
                        "prompt": prompt.strip(),
                        "style": style,
                        "png": data,
                        "model": info,
                    })
                    st.rerun()

    for i, item in enumerate(st.session_state.history_img):
        st.divider()

        st.image(
            item["png"],
            caption=(
                f'{item["prompt"]} — {item["style"]} '
                f'({item["model"]})'
            ),
            use_container_width=True,
        )

        st.download_button(
            "⬇️ Download PNG",
            data=item["png"],
            file_name=f"generated_image_{i + 1}.png",
            mime="image/png",
            key=f"download_img_{i}",
        )


# ============================================================
# MAIN STREAMLIT UI
# ============================================================

def main():
    st.set_page_config(
        page_title="AI Learning Studio",
        page_icon="🤖",
        layout="wide",
    )

    st.sidebar.title("🎓 AI Learning Studio")

    feature = st.sidebar.selectbox(
        "Choose feature",
        [
            "AI Teaching Assistant",
            "Math Mastermind",
            "Safe AI Image Generator",
        ],
    )

    provider = st.sidebar.selectbox(
        "AI provider",
        ["Groq", "Hugging Face"],
    )

    st.session_state["provider"] = provider

    if feature == "AI Teaching Assistant":
        run_ai_teaching_assistant()
    elif feature == "Math Mastermind":
        run_math_mastermind()
    else:
        run_safe_ai_image_generator()


if __name__ == "__main__":
    main()
'''

# ============================================================
# SAVE APP AND LAUNCH STREAMLIT
# ============================================================

with open("app.py", "w", encoding="utf-8") as file:
    file.write(APP_CODE)

print("app.py created.")

# Stop an old process occupying port 8501.
subprocess.run(
    ["bash", "-lc", "fuser -k 8501/tcp"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=False,
)

time.sleep(1)

# Start Streamlit and write logs to a file.
streamlit_log = open("streamlit.log", "w", encoding="utf-8")

streamlit_proc = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
    ],
    env=ENV,
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
)

print("Starting Streamlit...")

# Wait for Streamlit to become healthy.
ready = False

for _ in range(40):
    if streamlit_proc.poll() is not None:
        break

    try:
        with urllib.request.urlopen(
            "http://127.0.0.1:8501/_stcore/health",
            timeout=2,
        ) as response:
            if response.status == 200:
                ready = True
                break
    except Exception:
        pass

    time.sleep(1)

if not ready:
    streamlit_log.flush()
    print("Streamlit did not start. Recent log:")
    with open("streamlit.log", "r", encoding="utf-8") as file:
        print(file.read()[-5000:])

else:
    print("Streamlit is running.")

    # Download Cloudflare Tunnel if necessary.
    cloudflared_path = os.path.abspath("cloudflared")

    if not os.path.exists(cloudflared_path):
        print("Downloading Cloudflare Tunnel...")

        download = subprocess.run(
            [
                "wget", "-q", "-O", cloudflared_path,
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            ],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )

        if download.returncode != 0:
            print("Cloudflare download failed:")
            print(download.stdout[-3000:])

    if os.path.exists(cloudflared_path):
        os.chmod(cloudflared_path, 0o755)

        # Stop the previous tunnel created by this cell, if any.
        old_tunnel = globals().get("tunnel")

        if old_tunnel is not None and old_tunnel.poll() is None:
            old_tunnel.terminate()

            try:
                old_tunnel.wait(timeout=5)
            except subprocess.TimeoutExpired:
                old_tunnel.kill()

        tunnel_log = open("cloudflared.log", "w", encoding="utf-8")

        tunnel = subprocess.Popen(
            [
                cloudflared_path,
                "tunnel",
                "--no-autoupdate",
                "--url",
                "http://127.0.0.1:8501",
            ],
            stdout=tunnel_log,
            stderr=subprocess.STDOUT,
        )

        print("Starting Cloudflare Tunnel...")

        # Read the log file instead of blocking on readline().
        public_url = None

        for _ in range(45):
            if tunnel.poll() is not None:
                break

            try:
                with open(
                    "cloudflared.log",
                    "r",
                    encoding="utf-8",
                ) as file:
                    log_text = file.read()

                match = re.search(
                    r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
                    log_text,
                )

                if match:
                    public_url = match.group(0)
                    break

            except (OSError, UnicodeDecodeError):
                pass

            time.sleep(1)

        if public_url:
            print("\n" + "=" * 50)
            print("🚀 YOUR AI LEARNING STUDIO IS READY")
            print(public_url)
            print("=" * 50)
            print("Keep the Colab runtime running.")

        else:
            print("Cloudflare URL not detected.")
            print("Recent tunnel log:")
            with open("cloudflared.log", "r", encoding="utf-8") as file:
                print(file.read()[-5000:])
